# Vision Transformer: bars and spiral-arm tightness (JAX / Flax NNX)

**Hypothesis:** a ViT beats a CNN at detecting bars (`t03`) and estimating spiral-arm tightness
(`t10`).

This notebook is identical to `03_jax_cnn.ipynb` except for the model class and its learning
rate: same split, same input pipeline, same two-head masked loss, same optimizer recipe and
early stopping (all in `gz2/`). The last section compares the two models on the same test
galaxies.

**Design choices, from the EDA:**

- **Patch size 8, not the usual 16.** A typical labelled galaxy is only ~40 original SDSS pixels
  across. At a 128 px input, 16 px patches would give a coarse 8×8 grid; 8 px patches give 16×16
  = 256 tokens, fine enough to resolve a bar or arm structure.
- **Small model, from scratch.** About the same number of parameters as the CNN (printed below),
  so the comparison is about architecture rather than size. Both models train from scratch, so
  pretraining isn't a confound. ViTs are known to need more data than CNNs, and ~56k training
  galaxies is modest, which is part of what this experiment tests.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd
from flax import nnx

from gz2.evaluate import breakdown, load_context, load_predictions, metrics, paired_bootstrap, save_predictions
from gz2.jax_train import count_params, fit, predict
from gz2.pipeline import load_split, make_dataset

MODEL_NAME = "vit"
SEEDS = [0, 1, 2]
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 15
LR = 3e-4                   # transformers usually need a lower LR than CNNs; tune on validation
WARMUP_EPOCHS = 2           # transformers are sensitive to early large updates
LOSS_WEIGHTS = (1.0, 0.5)   # (bar, winding), same as the CNN
SMOKE_TEST = False

VIT_CONFIG = dict(patch_size=8, dim=128, depth=4, heads=4, mlp_dim=256, dropout=0.1)

print("JAX", jax.__version__, "| devices:", jax.devices())

## Data

Exactly the same split and pipeline as the CNN.

In [ ]:
split = load_split()
if SMOKE_TEST:
    split = split.groupby("split").sample(n=300, random_state=0)
    EPOCHS, SEEDS = 1, SEEDS[:1]

frames = {name: split[split["split"] == name] for name in ("train", "val", "test")}
val_ds = make_dataset(frames["val"], training=False, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
test_ds = make_dataset(frames["test"], training=False, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
print({name: len(f) for name, f in frames.items()})

## Model

A standard pre-norm ViT:

1. **Patch embedding:** a strided convolution cuts the image into 8×8 patches and projects each
   to a `dim`-length token.
2. A learned **[CLS] token** is prepended, and learned **position embeddings** are added, since
   attention by itself doesn't know where a patch came from.
3. **Encoder blocks:** LayerNorm → multi-head self-attention → residual, then LayerNorm → MLP →
   residual. Attention lets every patch look at every other patch from the first layer, which is
   the property the hypothesis is about (global structure like arms versus local texture).
4. The final [CLS] token feeds the same two heads as the CNN.

In [ ]:
class EncoderBlock(nnx.Module):
    def __init__(self, dim: int, heads: int, mlp_dim: int, dropout: float, *, rngs: nnx.Rngs):
        self.attn_norm = nnx.LayerNorm(dim, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(
            num_heads=heads, in_features=dim, dropout_rate=dropout, decode=False, deterministic=False, rngs=rngs
        )
        self.mlp_norm = nnx.LayerNorm(dim, rngs=rngs)
        self.mlp_in = nnx.Linear(dim, mlp_dim, rngs=rngs)
        self.mlp_out = nnx.Linear(mlp_dim, dim, rngs=rngs)
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        x = x + self.dropout(self.attn(self.attn_norm(x)))
        h = self.mlp_out(self.dropout(nnx.gelu(self.mlp_in(self.mlp_norm(x)))))
        return x + self.dropout(h)


class GalaxyViT(nnx.Module):
    def __init__(
        self,
        img_size: int = IMG_SIZE,
        patch_size: int = 8,
        dim: int = 128,
        depth: int = 4,
        heads: int = 4,
        mlp_dim: int = 256,
        dropout: float = 0.1,
        *,
        rngs: nnx.Rngs,
    ):
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nnx.Conv(
            3, dim, kernel_size=(patch_size, patch_size), strides=(patch_size, patch_size),
            padding="VALID", rngs=rngs,
        )
        self.cls_token = nnx.Param(jnp.zeros((1, 1, dim)))
        self.pos_embed = nnx.Param(0.02 * jax.random.normal(rngs.params(), (1, n_patches + 1, dim)))
        self.dropout = nnx.Dropout(dropout, rngs=rngs)
        self.blocks = nnx.List([EncoderBlock(dim, heads, mlp_dim, dropout, rngs=rngs) for _ in range(depth)])
        self.norm = nnx.LayerNorm(dim, rngs=rngs)
        self.bar_head = nnx.Linear(dim, 1, rngs=rngs)
        self.winding_head = nnx.Linear(dim, 3, rngs=rngs)

    def __call__(self, images: jax.Array) -> tuple[jax.Array, jax.Array]:
        x = self.patch_embed(images / 255.0)                 # (B, 16, 16, dim)
        batch, dim = x.shape[0], x.shape[-1]
        x = x.reshape(batch, -1, dim)                        # (B, 256 tokens, dim)
        cls = jnp.broadcast_to(self.cls_token[...], (batch, 1, dim))
        x = jnp.concatenate([cls, x], axis=1) + self.pos_embed[...]
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)[:, 0]                               # the [CLS] token
        return self.bar_head(x)[:, 0], self.winding_head(x)


model = GalaxyViT(**VIT_CONFIG, rngs=nnx.Rngs(0))
bar_logit, winding_logits = model(jnp.zeros((2, IMG_SIZE, IMG_SIZE, 3)))
print(f"{count_params(model):,} parameters | outputs: bar {bar_logit.shape}, winding {winding_logits.shape}")
print("Compare with the CNN's parameter count in 03_jax_cnn.ipynb; adjust dim/depth if they're far apart.")

## Train one model per seed

In [ ]:
histories, test_metrics = {}, {}
for seed in SEEDS:
    print(f"\n=== seed {seed} ===")
    train_ds = make_dataset(frames["train"], training=True, img_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=seed)
    model = GalaxyViT(**VIT_CONFIG, rngs=nnx.Rngs(params=seed, dropout=seed))
    best_model, histories[seed] = fit(
        model, train_ds, val_ds, epochs=EPOCHS, lr=LR, loss_weights=LOSS_WEIGHTS, warmup_epochs=WARMUP_EPOCHS
    )

    preds = predict(best_model, test_ds)
    test_metrics[seed] = metrics(preds, LOSS_WEIGHTS)
    print(f"saved -> {save_predictions(preds, MODEL_NAME, seed)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
panels = [("val_loss", "Val total loss"), ("val_bar_acc", "Val bar accuracy"), ("val_winding_rmse", "Val winding RMSE")]
for ax, (val_key, title) in zip(axes, panels):
    for seed, hist in histories.items():
        ax.plot(hist["epoch"], hist[val_key], linewidth=2, marker="o", markersize=4, label=f"seed {seed}")
    ax.set_title(title, loc="left")
    ax.set_xlabel("Epoch")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e6e5e0")
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

## Test results

In [ ]:
per_seed = pd.DataFrame(test_metrics).T[["bar_acc", "bar_auc", "winding_rmse", "winding_mae"]]
pd.concat([per_seed.rename(index=lambda s: f"seed {s}"), per_seed.agg(["mean", "std"])]).style.format("{:.4f}")

## CNN vs ViT

This needs `03_jax_cnn.ipynb` to have been run, so its predictions exist in
`output/predictions/`. Three views, from coarsest to most informative:

1. **Mean ± std over seeds.** Is the gap bigger than seed-to-seed noise?
2. **Paired bootstrap** on the same test galaxies: is the gap bigger than test-set noise? If the
   95% interval for *ViT − CNN* excludes 0, the difference is unlikely to be chance. For bar
   accuracy a positive difference favours the ViT; for winding RMSE a negative one does.
3. **Per galaxy-size bin.** Does the ViT win only on large, well-resolved galaxies? That would be
   a result in itself, given the EDA's finding that labels skew toward large galaxies.

In [ ]:
cnn_runs, vit_runs = load_predictions("jax_cnn"), load_predictions(MODEL_NAME)
if not cnn_runs:
    raise FileNotFoundError("No CNN predictions found: run 03_jax_cnn.ipynb first.")

seed_summary = pd.DataFrame({
    name: pd.DataFrame([metrics(p) for p in runs])[["bar_acc", "winding_rmse"]].agg(["mean", "std"]).unstack()
    for name, runs in {"CNN": cnn_runs, "ViT": vit_runs}.items()
}).T
seed_summary.columns = [f"{metric} {stat}" for metric, stat in seed_summary.columns]
print(f"{len(cnn_runs)} CNN seeds, {len(vit_runs)} ViT seeds")
seed_summary.style.format("{:.4f}")

In [ ]:
comparison = paired_bootstrap(cnn_runs, vit_runs).rename(columns={"A": "CNN", "B": "ViT", "B − A": "ViT − CNN"})
comparison.style.format({c: "{:.4f}" for c in comparison.columns if c != "n galaxies"} | {"n galaxies": "{:,.0f}"})

In [ ]:
context = load_context()
by_size = pd.concat(
    {"CNN": breakdown(cnn_runs[0], context), "ViT": breakdown(vit_runs[0], context)}, axis=1
)[[("CNN", "bar_acc"), ("ViT", "bar_acc"), ("CNN", "winding_rmse"), ("ViT", "winding_rmse")]]
by_size[("ViT − CNN", "bar_acc")] = by_size[("ViT", "bar_acc")] - by_size[("CNN", "bar_acc")]
by_size[("ViT − CNN", "winding_rmse")] = by_size[("ViT", "winding_rmse")] - by_size[("CNN", "winding_rmse")]
by_size.style.format("{:.4f}")

## Next steps

- **Tune each model's learning rate on validation** (e.g. {1e-4, 3e-4, 1e-3}) before the final
  comparison, so neither model is handicapped by a poor LR.
- **Patch size ablation:** 8 vs 16 tests the EDA's resolution argument directly.
- **Pretrained variant (separate experiment):** fine-tuning an ImageNet-pretrained ViT and CNN
  adds a different question, whether pretraining closes the ViT's data gap. Keep it separate from
  the from-scratch comparison.